# LA Airbnb: Phase 2 Clean Team Data for Travelers

**Project:** Los Angeles Airbnb listings · **Purpose:** canonical data and reusable analysis for the Phase 3 traveler dashboard.

This notebook is the readable, executable companion to [BUILD_SPEC.md](BUILD_SPEC.md). It prepares and inspects **all listings**, then demonstrates traveler-specific views. It does not change Phase 1.

**Traveler decision:** which neighborhoods and room types should I investigate, given my nightly budget and planned stay length?

**How to use:** open this notebook from the project folder, choose a Python environment with pandas, and **Restart Kernel → Run All**. Keep the notebook beside `LA_listings.csv` and the `phase2/` package. The full cleaned dataframe is named **`cleaned`**, with a typed reload named **`team_data`**. The canonical export is `phase2/outputs/listings.csv`.

**Team status:** proposed reconciled canonical version; team acceptance and integration into `main` are separate submission steps. Executed outputs establish data checks, not team approval.

## 1. Setup and source identity

The source is one listing snapshot, not a time series. Load every source field as text first to protect long identifiers and leading zeros. Paths are repository-relative; no downloads or machine-specific source paths are required.

In [1]:
from pathlib import Path
import sys
import json
import subprocess
import pandas as pd
from IPython.display import display, Markdown

search_roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in search_roots if (p / 'phase2/clean_data.py').is_file()
             and (p / 'LA_listings.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from the project folder or phase2 folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from phase2.clean_data import clean_listings, load_listings, fingerprint, IDS, BOOLS, COUNTS, FLOATS
from phase2.build import build
from phase2.backend import (Filters, dashboard_data, overview_metrics,
                           compare_neighbourhoods, rank_market_segments)

SOURCE = ROOT / 'LA_listings.csv'
SOURCE_HASH = fingerprint(SOURCE)
EXPECTED_SOURCE_HASH = '16007ed61afb75ffe44a69e7296bd50a71d61b2cce7cf885eb9d71d1dbd6f289'
assert SOURCE_HASH == EXPECTED_SOURCE_HASH, 'Source version changed: review definitions before proceeding.'
raw = pd.read_csv(SOURCE, dtype='string', keep_default_na=False)
display(pd.DataFrame([{'source': SOURCE.name, 'rows': len(raw), 'source_columns': len(raw.columns),
                       'collection_date': 'Unknown', 'source_sha256': SOURCE_HASH}]))

,source,rows,source_columns,collection_date,source_sha256
0,LA_listings.csv,43932,19,Unknown,16007ed61afb75ffe44a69e7296bd50a71d61b2cce7cf8...


## 2. Inspect the raw data

**Grain:** one listing per `id`. Repeated hosts can have different listings and are retained. The source includes four room types and neighborhood categories; room type is **not** bedroom count. Snapshot and download dates are unknown for these exact source bytes.

The preview contains only fields needed for traveler analysis. Host names and license strings are unnecessary for dashboard display.

In [2]:
TRAVELER_COLUMNS = ['id', 'neighbourhood_group', 'neighbourhood', 'room_type', 'price',
                   'minimum_nights', 'number_of_reviews', 'number_of_reviews_ltm',
                   'last_review', 'reviews_per_month', 'availability_365']
display(raw[TRAVELER_COLUMNS].head(8))
display(pd.DataFrame({
    'raw_type': raw.dtypes.astype(str),
    'raw_blank_n': [int(raw[c].str.strip().eq('').sum()) for c in raw],
    'distinct_nonblank_values': [int(raw.loc[raw[c].str.strip().ne(''), c].nunique()) for c in raw]
}))
display(raw.room_type.value_counts().rename_axis('room_type').to_frame('listing_count'))
display(pd.DataFrame([{'exact_raw_duplicate_rows': int(raw.duplicated().sum()),
                       'duplicate_listing_id_rows': int(raw.id.duplicated(keep=False).sum())}]))

,id,neighbourhood_group,neighbourhood,room_type,price,minimum_nights,number_of_reviews,number_of_reviews_ltm,last_review,reviews_per_month,availability_365
0,2708,City of Los Angeles,Hollywood,Private room,67,30,47,3,2026-05-26,0.32,301
1,2732,Other Cities,Santa Monica,Private room,214,1,24,0,2022-08-21,0.13,350
2,6033,City of Los Angeles,Woodland Hills,Entire home/apt,100,30,19,0,2020-03-22,0.09,270
3,6931,City of Los Angeles,Hollywood,Private room,95,30,39,2,2025-12-15,0.19,259
4,7874,Other Cities,Bellflower,Private room,113,1,26,9,2026-04-02,0.29,103
5,8021,City of Los Angeles,Pico-Robertson,Private room,96,31,238,0,2025-04-27,1.15,298
6,9140,Other Cities,Long Beach,Private room,,31,389,0,2019-11-25,2.40,0
7,9376,City of Los Angeles,Venice,Private room,199,30,54,0,2022-10-25,0.26,365


,raw_type,raw_blank_n,distinct_nonblank_values
id,string,0,43932
name,string,0,42715
host_id,string,181,21024
host_profile_id,string,395,20969
host_name,string,384,8024
neighbourhood_group,string,0,3
neighbourhood,string,0,265
latitude,string,0,33397
longitude,string,0,35510
room_type,string,0,4


,listing_count
room_type,
Entire home/apt,32454
Private room,10771
Hotel room,415
Shared room,292


,exact_raw_duplicate_rows,duplicate_listing_id_rows
0,0,0


## 3. Agreed-definition proposal and cleaning rules

The team branches differed on row deletion, review-rate filling, text cleanup, source version and outlier thresholds.

| Issue | Canonical Phase 2 rule | Why it helps the traveler dashboard |
|---|---|---|
| Unit of analysis | One unique listing `id`; remove only exact normalized duplicates; stop on conflicting IDs | Avoid double counting or arbitrary record selection |
| Identifier types | Keep all three ID columns as strings | Prevent irreversible loss of digits |
| Text/category consistency | Remove invisible formatting, normalize whitespace; preserve accents, punctuation and case | Consistent labels without inventing aliases |
| Missing prices | Keep listings and null prices; disclose price denominator | Preserve inventory without inventing affordability |
| Minimum stay | Keep nulls; apply minimum_nights <= traveler trip length only in analysis | Separate cleaning from traveler eligibility |
| Review rates | Keep missing rates null; preserve true zero review counts | Avoid treating an unobserved rate as measured zero |
| Extreme prices | Keep positive values, flag > Q3 + 3×IQR once on full source | Medians remain primary; optional sensitivity is explicit |
| Invalid values | Stop on impossible or unparseable numbers/dates | Surface source problems rather than silently deleting rows |
| Missing licenses | Blank/N/A becomes null; no compliance claim | Avoid unsupported conclusions |
| Coordinates | Flag missing/out-of-screen coordinates; omit from map only | Retain nongeographic evidence |
| Joins | None; only the supplied listing source | No fabricated boundary, booking or amenity data |

Implementation lives in `phase2/clean_data.py`, so the notebook and backend share one cleaning definition. See [DATA_CONTRACT.md](phase2/DATA_CONTRACT.md) for all source and derived fields.

In [3]:
cleaned, cleaning_audit = clean_listings(raw)
assert fingerprint(SOURCE) == SOURCE_HASH
display(pd.DataFrame([{
    'input_rows': len(raw), 'cleaned_rows': len(cleaned),
    'source_columns': len(raw.columns), 'cleaned_columns': len(cleaned.columns),
    'exact_duplicates_removed': cleaning_audit['exact_duplicates_removed'],
    'unique_listing_ids': cleaned.id.nunique(),
    'source_unchanged': fingerprint(SOURCE) == SOURCE_HASH
}]))
display(cleaned[TRAVELER_COLUMNS].head(10))

,input_rows,cleaned_rows,source_columns,cleaned_columns,exact_duplicates_removed,unique_listing_ids,source_unchanged
0,43932,43932,19,31,0,43932,True


,id,neighbourhood_group,neighbourhood,room_type,price,minimum_nights,number_of_reviews,number_of_reviews_ltm,last_review,reviews_per_month,availability_365
0,2708,City of Los Angeles,Hollywood,Private room,67.0,30,47,3,2026-05-26,0.32,301
1,2732,Other Cities,Santa Monica,Private room,214.0,1,24,0,2022-08-21,0.13,350
2,6033,City of Los Angeles,Woodland Hills,Entire home/apt,100.0,30,19,0,2020-03-22,0.09,270
3,6931,City of Los Angeles,Hollywood,Private room,95.0,30,39,2,2025-12-15,0.19,259
4,7874,Other Cities,Bellflower,Private room,113.0,1,26,9,2026-04-02,0.29,103
5,8021,City of Los Angeles,Pico-Robertson,Private room,96.0,31,238,0,2025-04-27,1.15,298
6,9140,Other Cities,Long Beach,Private room,<NA>,31,389,0,2019-11-25,2.4,0
7,9376,City of Los Angeles,Venice,Private room,199.0,30,54,0,2022-10-25,0.26,365
8,11877,City of Los Angeles,Venice,Entire home/apt,115.0,30,46,0,2021-06-01,0.26,74
9,15089,City of Los Angeles,Mid-City,Entire home/apt,130.0,30,98,2,2025-12-01,0.48,351


## 4. Data types, missingness and derived flags

Missingness belongs to the metric denominator, not an automatic row deletion. Blank/N/A licenses normalize to null; raw blank counts alone omit the four literal `N/A` license markers. Missing monthly review rates on never-reviewed listings remain null.

In [4]:
profile = pd.DataFrame({
    'clean_type': cleaned.dtypes.astype(str),
    'missing_n': cleaned.isna().sum(),
    'missing_percent': (cleaned.isna().mean() * 100).round(2)
})
display(profile)
missingness = pd.DataFrame({
    'raw_blank_n': [int(raw[c].str.strip().eq('').sum()) for c in raw],
    'clean_missing_n': [int(cleaned[c].isna().sum()) for c in raw]
}, index=raw.columns)
display(missingness.loc[(missingness.raw_blank_n > 0) | (missingness.clean_missing_n > 0)])
display(pd.Series(cleaning_audit['flags'], name='flagged_listings').to_frame())
display(pd.Series(cleaning_audit['text_changes'], name='normalized_text_rows').to_frame())
print(f"Full-source high-price flag threshold: ${cleaning_audit['high_price_threshold_usd']:,.0f}/night")

,clean_type,missing_n,missing_percent
id,string,0,0.00
name,string,0,0.00
host_id,string,181,0.41
host_profile_id,string,395,0.90
host_name,string,384,0.87
neighbourhood_group,string,0,0.00
neighbourhood,string,0,0.00
latitude,Float64,0,0.00
longitude,Float64,0,0.00
room_type,string,0,0.00


,raw_blank_n,clean_missing_n
host_id,181,181
host_profile_id,395,395
host_name,384,384
price,5570,5570
minimum_nights,11,11
last_review,11311,11311
reviews_per_month,11311,11311
calculated_host_listings_count,181,181
license,30736,30740


,flagged_listings
price_missing,5570
minimum_nights_missing,11
never_reviewed,11311
review_fields_inconsistent,0
review_counts_inconsistent,0
price_high_outlier,1634
price_under_10,14
minimum_nights_over_365,24
coordinate_review,0
license_missing,30740


,normalized_text_rows
name,1070
host_name,257
neighbourhood_group,0
neighbourhood,0
room_type,0
license,6


Full-source high-price flag threshold: $1,276/night


## 5. Independent row and value reconciliation

Check every identifier and every observed numeric value, not just a preview. All 43,932 source listings must remain in the canonical data for this source; traveler filters are applied later. Date syntax and numeric domain validation are performed by the cleaner.

In [5]:
checks = []
def record_check(label, passed):
    checks.append({'check': label, 'passed': bool(passed)})

record_check('All source listings retained', len(raw) == len(cleaned) == 43932)
record_check('Unique, nonmissing listing ID', cleaned.id.is_unique and cleaned.id.notna().all())
for column in IDS:
    record_check(f'{column}: every identifier digit preserved',
                 raw[column].str.strip().tolist() == cleaned[column].fillna('').tolist())
for column in COUNTS + FLOATS:
    before = pd.to_numeric(raw[column], errors='coerce').astype('Float64')
    after = cleaned[column].astype('Float64')
    pd.testing.assert_series_equal(before, after, check_names=False)
    record_check(f'{column}: observed values and nulls unchanged', True)
record_check('Zero-review monthly rates remain null',
             cleaned.loc[cleaned.never_reviewed, 'reviews_per_month'].isna().all())
record_check('Source bytes unchanged', fingerprint(SOURCE) == SOURCE_HASH)
record_checks = pd.DataFrame(checks)
display(record_checks)
assert record_checks.passed.all()

,check,passed
0,All source listings retained,True
1,"Unique, nonmissing listing ID",True
2,id: every identifier digit preserved,True
3,host_id: every identifier digit preserved,True
4,host_profile_id: every identifier digit preserved,True
5,minimum_nights: observed values and nulls unch...,True
6,number_of_reviews: observed values and nulls u...,True
7,calculated_host_listings_count: observed value...,True
8,availability_365: observed values and nulls un...,True
9,number_of_reviews_ltm: observed values and nul...,True


## 6. Inspect real difficult cases

The rows below explain how source values become cleaned values. The examples are selected from actual conditions, not invented data. High-price listings remain in the dataset. IDs remain strings throughout.

In [6]:
case_rows = []
case_definitions = [
    ('Missing price retained', cleaned.price_missing, 'price'),
    ('Unknown minimum stay retained', cleaned.minimum_nights_missing, 'minimum_nights'),
    ('Never-reviewed rate stays null', cleaned.never_reviewed, 'reviews_per_month'),
    ('High price retained and flagged', cleaned.price_high_outlier, 'price'),
    ('Large secondary identifier preserved', cleaned.host_profile_id.str.len().gt(16).fillna(False), 'host_profile_id'),
    ('Missing license remains unknown', cleaned.license_missing, 'license'),
]
for explanation, condition, column in case_definitions:
    idx = cleaned.index[condition][0]
    case_rows.append({'reason': explanation, 'listing_id': cleaned.at[idx, 'id'], 'column': column,
                      'raw_value': repr(raw.at[idx, column]),
                      'clean_value': repr(cleaned.at[idx, column])})
display(pd.DataFrame(case_rows))

,reason,listing_id,column,raw_value,clean_value
0,Missing price retained,9140,price,'',<NA>
1,Unknown minimum stay retained,2436587,minimum_nights,'',<NA>
2,Never-reviewed rate stays null,584724,reviews_per_month,'',<NA>
3,High price retained and flagged,98284,price,'2549',np.float64(2549.0)
4,Large secondary identifier preserved,2708,host_profile_id,'1462506333679450606','1462506333679450606'
5,Missing license remains unknown,2708,license,'',<NA>


## 7. Export and reload the canonical team dataset

The build function regenerates the canonical CSV, source audit, missingness table, dashboard examples and hash manifest from the same source. Typed reloading is essential: ordinary CSV inference can corrupt identifiers or interpret booleans incorrectly.

The full dataframe is `team_data`. CSV output is `phase2/outputs/listings.csv`; it has **43,932 rows and 31 columns**. Do not substitute earlier project output copies.

In [7]:
build_audit = build()
team_data = load_listings()
pd.testing.assert_frame_equal(cleaned, team_data, check_dtype=False)
assert all(str(team_data[c].dtype) == 'string' for c in IDS)
assert fingerprint(SOURCE) == SOURCE_HASH
display(pd.DataFrame([{'canonical_file': 'phase2/outputs/listings.csv',
                       'rows': len(team_data), 'columns': len(team_data.columns),
                       'export_roundtrip_passed': True}]))
display(team_data[TRAVELER_COLUMNS + ['price_missing', 'minimum_nights_missing',
                                     'price_high_outlier']].head(10))

,canonical_file,rows,columns,export_roundtrip_passed
0,phase2/outputs/listings.csv,43932,31,True


,id,neighbourhood_group,neighbourhood,room_type,price,minimum_nights,number_of_reviews,number_of_reviews_ltm,last_review,reviews_per_month,availability_365,price_missing,minimum_nights_missing,price_high_outlier
0,2708,City of Los Angeles,Hollywood,Private room,67.0,30,47,3,2026-05-26,0.32,301,False,False,False
1,2732,Other Cities,Santa Monica,Private room,214.0,1,24,0,2022-08-21,0.13,350,False,False,False
2,6033,City of Los Angeles,Woodland Hills,Entire home/apt,100.0,30,19,0,2020-03-22,0.09,270,False,False,False
3,6931,City of Los Angeles,Hollywood,Private room,95.0,30,39,2,2025-12-15,0.19,259,False,False,False
4,7874,Other Cities,Bellflower,Private room,113.0,1,26,9,2026-04-02,0.29,103,False,False,False
5,8021,City of Los Angeles,Pico-Robertson,Private room,96.0,31,238,0,2025-04-27,1.15,298,False,False,False
6,9140,Other Cities,Long Beach,Private room,<NA>,31,389,0,2019-11-25,2.4,0,True,False,False
7,9376,City of Los Angeles,Venice,Private room,199.0,30,54,0,2022-10-25,0.26,365,False,False,False
8,11877,City of Los Angeles,Venice,Entire home/apt,115.0,30,46,0,2021-06-01,0.26,74,False,False,False
9,15089,City of Los Angeles,Mid-City,Entire home/apt,130.0,30,98,2,2025-12-01,0.48,351,False,False,False


## 8. Traveler scenario and metric definitions

Change the settings in the next cell and rerun subsequent analysis cells. The example is a **7-night stay, entire home/apartment, $250/night budget**. These are analytical controls, not a live booking query.

| Traveler metric | Definition / interpretation |
|---|---|
| Matching listings | Selected room type; minimum stay <= planned nights; price <= budget when set |
| Typical nightly price | Median of observed prices, with price_n and middle 50% range |
| Within-budget share | Count price <= budget divided by **pre-budget** observed-price count |
| Review history | Total and source last-12-month review counts, including real zeros; not ratings |
| Calendar context | Source annual available nights; does not establish availability for travel dates |

Unknown minimum stay is excluded when a specific trip length is active. With no budget, missing-price listings remain in counts but not price statistics. No bedrooms, bathrooms, guest capacity or date filters are possible from this source.

In [8]:
TRIP_NIGHTS = 7
NIGHTLY_BUDGET_USD = 250
ROOM_TYPE = 'Entire home/apt'
MIN_TOTAL_REVIEWS = 0
EXCLUDE_FLAGGED_HIGH_PRICES = False

traveler_filters = Filters(trip_nights=TRIP_NIGHTS, budget_usd=NIGHTLY_BUDGET_USD,
                          room_types=(ROOM_TYPE,), min_reviews=MIN_TOTAL_REVIEWS,
                          exclude_high_prices=EXCLUDE_FLAGGED_HIGH_PRICES)
views = dashboard_data(team_data, traveler_filters)
display(Markdown('**Selected population: listings meeting the nightly budget and all other controls.**'))
display(views['overview'])
display(Markdown('**Affordability population: same controls before applying the nightly budget.**'))
display(views['affordability'])
affordability = views['affordability'].iloc[0]
selected = views['overview'].iloc[0]
if NIGHTLY_BUDGET_USD is not None:
    assert affordability.within_budget_n == selected.listing_count
    if affordability.price_n:
        display(Markdown(
            f"**{int(affordability.within_budget_n):,} of {int(affordability.price_n):,} priced listings "
            f"({affordability.within_budget_share:.1%})** fit the nightly budget. "
            f"The pre-budget population also contains {int(affordability.missing_price_n):,} listings without a price."
        ))

**Selected population: listings meeting the nightly budget and all other controls.**

,listing_count,price_n,missing_price_n,median_price,price_p25,price_p75,within_budget_n,within_budget_share,review_n,median_total_reviews,review_ltm_n,median_reviews_ltm,reviewed_share,review_rate_n,mean_observed_reviews_per_month,availability_n,median_available_nights
0,3773,3773,0,194.0,159.0,225.0,None,NaN,3773,47.0,3773,18.0,0.927644,3500,2.693809,3773,247.0


**Affordability population: same controls before applying the nightly budget.**

,listing_count,price_n,missing_price_n,median_price,price_p25,price_p75,within_budget_n,within_budget_share,review_n,median_total_reviews,review_ltm_n,median_reviews_ltm,reviewed_share,review_rate_n,mean_observed_reviews_per_month,availability_n,median_available_nights
0,16633,15412,1221,371.0,252.0,580.0,3773,0.244809,16633,26.0,16633,11.0,0.889497,14795,2.158351,16633,241.0


**3,773 of 15,412 priced listings (24.5%)** fit the nightly budget. The pre-budget population also contains 1,221 listings without a price.

## 9. Neighborhood shortlist with sample sizes

Rank only within the selected room type. A minimum **30 observed prices** is a display rule, not a claim of statistical significance. Low-sample areas remain in the complete evidence table. Price medians are calculated from listing rows, never averaged across neighborhood medians.

The affordability table uses pre-budget data; otherwise applying the budget first would force every share to 100%.

In [9]:
price_ranking = rank_market_segments(views['segments'], metric='median_price', min_n=30)
display(price_ranking[['rank', 'neighbourhood', 'room_type', 'listing_count', 'price_n',
                       'median_price', 'price_p25', 'price_p75', 'missing_price_n']].head(15))
if NIGHTLY_BUDGET_USD is not None:
    affordability_ranking = rank_market_segments(
        views['affordability_segments'], metric='within_budget_share', min_n=30, ascending=False)
    display(affordability_ranking[['rank', 'neighbourhood', 'room_type', 'price_n',
                                   'within_budget_n', 'within_budget_share']].head(15))
display(Markdown('**Complete segment evidence, including small groups (first 20 rows shown):**'))
display(views['segments'].assign(small_price_sample=lambda d: d.price_n < 30).head(20))

,rank,neighbourhood,room_type,listing_count,price_n,median_price,price_p25,price_p75,missing_price_n
0,1,El Monte,Entire home/apt,106,106,134.5,120.00,177.75,0
1,2,Lancaster,Entire home/apt,86,86,146.0,105.25,205.50,0
2,3,Baldwin Park,Entire home/apt,63,63,153.0,123.50,185.00,0
3,4,Rowland Heights,Entire home/apt,71,71,167.0,137.50,206.00,0
4,5,East San Gabriel,Entire home/apt,43,43,172.0,136.50,203.00,0
5,6,Torrance,Entire home/apt,57,57,177.0,155.00,206.00,0
6,7,Monterey Park,Entire home/apt,36,36,182.0,164.75,212.25,0
7,8,Hacienda Heights,Entire home/apt,42,42,185.0,153.25,218.00,0
8,9,Downey,Entire home/apt,36,36,185.5,162.00,216.25,0
9,10,Alhambra,Entire home/apt,146,146,189.0,163.00,226.00,0


,rank,neighbourhood,room_type,price_n,within_budget_n,within_budget_share
0,1,Baldwin Park,Entire home/apt,91,63,0.692308
1,2,El Monte,Entire home/apt,168,106,0.630952
2,3,East San Gabriel,Entire home/apt,71,43,0.605634
3,4,Lancaster,Entire home/apt,144,86,0.597222
4,5,Carson,Entire home/apt,48,28,0.583333
5,6,Whittier,Entire home/apt,75,43,0.573333
6,7,Norwalk,Entire home/apt,44,23,0.522727
7,8,South Gate,Entire home/apt,43,22,0.511628
8,9,Pomona,Entire home/apt,63,32,0.507937
9,10,Gardena,Entire home/apt,52,26,0.500000


**Complete segment evidence, including small groups (first 20 rows shown):**

,neighbourhood,room_type,listing_count,price_n,median_price,price_p25,price_p75,within_budget_n,within_budget_share,median_total_reviews,median_reviews_ltm,reviewed_share,review_n,review_ltm_n,review_rate_n,mean_observed_reviews_per_month,availability_n,median_available_nights,missing_price_n,small_price_sample
0,Acton,Entire home/apt,5,5,139.0,128.00,175.00,None,NaN,12.0,12.0,1.000000,5,5,5,1.880000,5,346.0,0,True
1,Adams-Normandie,Entire home/apt,2,2,103.0,93.50,112.50,None,NaN,59.5,16.5,0.500000,2,2,1,3.630000,2,199.5,0,True
2,Agoura Hills,Entire home/apt,2,2,104.0,73.00,135.00,None,NaN,107.0,9.0,0.500000,2,2,1,3.770000,2,316.0,0,True
3,Agua Dulce,Entire home/apt,8,8,168.5,158.50,183.25,None,NaN,5.0,3.5,0.875000,8,8,7,1.208571,8,339.5,0,True
4,Alhambra,Entire home/apt,146,146,189.0,163.00,226.00,None,NaN,57.0,18.0,0.917808,146,146,134,2.665597,146,246.0,0,False
5,Alondra Park,Entire home/apt,5,5,223.0,199.00,227.00,None,NaN,158.0,22.0,0.800000,5,5,4,5.257500,5,312.0,0,True
6,Altadena,Entire home/apt,23,23,198.0,152.50,223.00,None,NaN,52.0,2.0,0.869565,23,23,20,1.594500,23,133.0,0,True
7,Angeles Crest,Entire home/apt,2,2,235.0,228.00,242.00,None,NaN,22.0,4.5,1.000000,2,2,2,0.800000,2,307.5,0,True
8,Arcadia,Entire home/apt,28,28,185.5,128.00,219.25,None,NaN,11.5,9.5,0.964286,28,28,27,1.795556,28,175.0,0,True
9,Arlington Heights,Entire home/apt,3,3,177.0,135.50,177.50,None,NaN,473.0,45.0,1.000000,3,3,3,4.816667,3,67.0,0,True


## 10. Like-for-like neighborhood comparison

Compare Hollywood and Venice under the **same room type, stay length, review filter and price-sensitivity settings**. Price and affordability evidence use the pre-budget population. The budget is a threshold for the affordability share, not a restriction that hides the price distribution of either area.

In [10]:
comparison = compare_neighbourhoods(team_data, traveler_filters, ('Hollywood', 'Venice'))
display(comparison[['neighbourhood', 'listing_count', 'price_n', 'median_price', 'price_p25',
                    'price_p75', 'within_budget_n', 'within_budget_share', 'median_total_reviews',
                    'median_reviews_ltm', 'review_ltm_n']])
display(Markdown('Use the observed sample sizes when assessing differences. These comparisons do not establish a better guest experience or current booking availability.'))

,neighbourhood,listing_count,price_n,median_price,price_p25,price_p75,within_budget_n,within_budget_share,median_total_reviews,median_reviews_ltm,review_ltm_n
0,Hollywood,337,327,336.0,264.00,481.0,69,0.211009,16.0,11.0,337
1,Venice,402,374,590.0,388.25,960.5,24,0.064171,50.0,14.0,402


Use the observed sample sizes when assessing differences. These comparisons do not establish a better guest experience or current booking availability.

## 11. Data ready for dashboard visualizations

| Phase 3 visual | Backend table / design |
|---|---|
| Color-coded horizontal price bars | `views['segments']`, ranked with price_n >=30; USD axis and sample-size tooltips |
| Within-budget bars | `views['affordability_segments']`; 0–100% axis, pre-budget denominator |
| Step line | `views['price_distribution']`: cumulative share by listed price, **not a time trend** |
| Geographic hover markers | `views['map']` or `views['affordability_map']`; approximate neighborhood centers, count/price/sample-size hover |
| Comparison and evidence tables | Same-room neighborhood comparison and paginated listing details |

No geographic boundary file is present, so marker locations must not be presented as official region centroids or boundaries. Color scales, hover fields, axes, reset behavior and empty states are specified in BUILD_SPEC. The interactive interface is Phase 3.

In [11]:
display(Markdown('**Price distribution: exact observed price points, pre-budget population.**'))
display(views['price_distribution'].head(10))
display(Markdown('**Geographic affordability evidence: pre-budget population, approximate centers.**'))
display(views['affordability_map'][['neighbourhood', 'room_type', 'latitude', 'longitude',
                                    'map_n', 'listing_count', 'price_n', 'median_price',
                                    'within_budget_n', 'within_budget_share']].head(10))
display(Markdown('**Traveler listing details: first 15 selected listings, price ascending.**'))
display(views['listings'].sort_values(['price', 'id'], na_position='last')[TRAVELER_COLUMNS].head(15))

**Price distribution: exact observed price points, pre-budget population.**

,room_type,nightly_price_usd,listing_count_at_price,price_n,cumulative_share
0,Entire home/apt,16.0,2,15412,0.000130
1,Entire home/apt,23.0,1,15412,0.000195
2,Entire home/apt,36.0,1,15412,0.000260
3,Entire home/apt,42.0,1,15412,0.000324
4,Entire home/apt,55.0,1,15412,0.000389
5,Entire home/apt,56.0,1,15412,0.000454
6,Entire home/apt,60.0,4,15412,0.000714
7,Entire home/apt,61.0,1,15412,0.000779
8,Entire home/apt,62.0,1,15412,0.000843
9,Entire home/apt,63.0,1,15412,0.000908


**Geographic affordability evidence: pre-budget population, approximate centers.**

,neighbourhood,room_type,latitude,longitude,map_n,listing_count,price_n,median_price,within_budget_n,within_budget_share
0,Acton,Entire home/apt,34.501086,-118.207474,15,15,12,292.5,5,0.416667
1,Adams-Normandie,Entire home/apt,34.031539,-118.298466,4,4,4,210.5,2,0.500000
2,Agoura Hills,Entire home/apt,34.151497,-118.767695,23,23,20,544.5,2,0.100000
3,Agua Dulce,Entire home/apt,34.50282,-118.33117,16,16,16,233.5,8,0.500000
4,Alhambra,Entire home/apt,34.084406,-118.134117,443,443,402,283.0,146,0.363184
5,Alondra Park,Entire home/apt,33.892333,-118.333875,16,16,15,312.0,5,0.333333
6,Altadena,Entire home/apt,34.187236,-118.140058,83,83,68,299.5,23,0.338235
7,Angeles Crest,Entire home/apt,34.273064,-117.930813,6,6,6,444.5,2,0.333333
8,Arcadia,Entire home/apt,34.123016,-118.04119,70,70,67,276.0,28,0.417910
9,Arlington Heights,Entire home/apt,34.045339,-118.322102,13,13,13,426.0,3,0.230769


**Traveler listing details: first 15 selected listings, price ascending.**

,id,neighbourhood_group,neighbourhood,room_type,price,minimum_nights,number_of_reviews,number_of_reviews_ltm,last_review,reviews_per_month,availability_365
37469,1577809077219535071,Other Cities,Lancaster,Entire home/apt,16.0,5,0,0,<NA>,<NA>,269
19812,905128298331494648,Unincorporated Areas,Northeast Antelope Valley,Entire home/apt,16.0,1,3,1,2026-02-10,0.11,365
23355,1080631842692336432,Unincorporated Areas,Southeast Antelope Valley,Entire home/apt,23.0,1,1,0,2024-08-02,0.04,360
10400,45127845,Other Cities,Santa Clarita,Entire home/apt,36.0,2,40,0,2024-06-30,0.57,20
15442,662248128294833398,Other Cities,Agoura Hills,Entire home/apt,42.0,7,0,0,<NA>,<NA>,358
43491,1701970788689742273,Other Cities,Palmdale,Entire home/apt,55.0,1,0,0,<NA>,<NA>,357
22795,1071158678430122336,Unincorporated Areas,Sun Village,Entire home/apt,56.0,2,106,23,2026-05-20,3.67,262
24034,1119142267962370137,Unincorporated Areas,Sun Village,Entire home/apt,60.0,2,45,6,2025-10-04,1.74,344
25871,1203191054449012525,Unincorporated Areas,Sun Village,Entire home/apt,60.0,2,52,7,2026-03-31,2.25,350
38283,1592290569169549697,Unincorporated Areas,Northeast Antelope Valley,Entire home/apt,60.0,3,2,2,2026-02-16,0.43,318


## 12. Automated validation and reproducibility

Run the project backend suite in a separate process. It checks identifier/numeric preservation, null handling, budget and stay-length boundaries, aggregation denominators, ranking, geographic exclusions, price distributions, comparisons and byte-identical output regeneration.

This fresh-process test is **not** the independent fresh-agent review required before submission.

In [12]:
result = subprocess.run([sys.executable, '-m', 'phase2.test_backend'], cwd=ROOT,
                        text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Backend verification failed; review the output above.'
verification = json.loads((ROOT / 'phase2/outputs/verification.json').read_text())
display(pd.DataFrame([{k: verification[k] for k in ['passed', 'tests_run', 'failures', 'errors']}]))
assert fingerprint(SOURCE) == SOURCE_HASH


test_aggregation_weighting_and_ranking_denominator (__main__.BackendTests) ... ok
test_bad_controls (__main__.BackendTests) ... ok
test_cleaner_rejects_conflicting_ids_and_float_ids (__main__.BackendTests) ... ok
test_exact_stay_boundary_missing_prices_and_coordinate_screen (__main__.BackendTests) ... ok
test_fresh_process_rebuild (__main__.BackendTests) ... ok
test_full_source_preserved (__main__.BackendTests) ... ok
test_geography_and_cdf (__main__.BackendTests) ... ok
test_like_for_like_comparison (__main__.BackendTests) ... ok
test_nulls_not_zero_and_empty_contract (__main__.BackendTests) ... ok
test_traveler_trip_and_budget_independently (__main__.BackendTests) ... ok

----------------------------------------------------------------------
Ran 10 tests in 5.739s

OK



,passed,tests_run,failures,errors
0,True,10,0,0
